# Coconut Mite Detection Model v12
## Coconut Fruit Surface Pattern Recognition

**Focus:** Detect mite damage patterns on the coconut fruit (pol gediya) surface

**Key Improvements over v10/v11:**
- Binary classification: `mite` vs `healthy` (no not_coconut - dataset contains only coconut fruits)
- Texture-focused preprocessing and augmentation
- Center-weighted augmentation (coconut usually in center)
- Enhanced data augmentation for fruit surface patterns
- Uses new high-quality dataset with clear mite damage patterns

**Dataset:** `mite - new` (1380 mite, 700 healthy coconut fruit images)

**Mite Damage Characteristics:**
- Brown/dark patches on fruit surface
- Scaly/cross-hatch texture pattern
- Surface damage spreading from specific areas

## 1. Setup and Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import shutil
import warnings
warnings.filterwarnings('ignore')
from PIL import Image
import random

# TensorFlow imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Sklearn imports for metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    precision_recall_fscore_support,
    accuracy_score,
    roc_auc_score,
    roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

## 2. Configuration

In [ ]:
# Paths
BASE_DIR = r'D:\SLIIT\Reaserch Project\CoconutHealthMonitor\Research\ml'
RAW_DATA_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'mite - new')
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, 'data', 'processed', 'mite_v12')
MODEL_DIR = os.path.join(BASE_DIR, 'models', 'coconut_mite_v12')

# Create directories
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

# Training configuration - optimized for texture recognition
CONFIG = {
    'img_size': 224,
    'batch_size': 32,
    'epochs': 60,
    'learning_rate': 0.0001,
    'patience': 15,
    'min_delta': 0.001,
    'l2_reg': 0.01,
    'dropout': 0.4,
    'focal_gamma': 2.0,
    'focal_alpha': 0.25,
    'train_split': 0.7,
    'val_split': 0.15,
    'test_split': 0.15,
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Data Exploration

In [ ]:
def count_images(directory):
    """Count images in each class folder"""
    counts = {}
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            count = len([f for f in os.listdir(class_path) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            counts[class_name] = count
    return counts

# Count raw images
raw_counts = count_images(RAW_DATA_DIR)

print("=" * 50)
print("RAW DATASET SUMMARY")
print("=" * 50)
total = 0
for class_name, count in raw_counts.items():
    print(f"  {class_name}: {count} images")
    total += count
print(f"  TOTAL: {total} images")
print("=" * 50)

# Note: 'healty' is misspelled in folder name, will fix during processing

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row, class_name in enumerate(['mite', 'healty']):
    class_path = os.path.join(RAW_DATA_DIR, class_name)
    images = os.listdir(class_path)[:4]
    
    for col, img_name in enumerate(images):
        img_path = os.path.join(class_path, img_name)
        img = Image.open(img_path)
        
        axes[row, col].imshow(img)
        display_name = 'healthy' if class_name == 'healty' else class_name
        axes[row, col].set_title(f'{display_name.upper()}', fontsize=12, fontweight='bold')
        axes[row, col].axis('off')

plt.suptitle('Sample Images: Mite vs Healthy Coconut Fruits', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'sample_images.png'), dpi=150)
plt.show()

## 4. Data Splitting (Train/Val/Test)

In [ ]:
def create_split_directories(base_dir):
    """Create train/validation/test directories"""
    for split in ['train', 'validation', 'test']:
        for class_name in ['mite', 'healthy']:
            path = os.path.join(base_dir, split, class_name)
            os.makedirs(path, exist_ok=True)
    print("Split directories created!")

def split_and_copy_data(raw_dir, processed_dir, train_ratio=0.7, val_ratio=0.15):
    """Split data into train/val/test and copy to new location"""
    
    # Mapping for folder names (fix 'healty' typo)
    folder_mapping = {'mite': 'mite', 'healty': 'healthy'}
    
    stats = {'train': {}, 'validation': {}, 'test': {}}
    
    for raw_name, clean_name in folder_mapping.items():
        class_path = os.path.join(raw_dir, raw_name)
        
        # Get all images
        images = [f for f in os.listdir(class_path) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        # Shuffle for random split
        random.shuffle(images)
        
        # Calculate split indices
        n = len(images)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))
        
        # Split
        train_images = images[:train_end]
        val_images = images[train_end:val_end]
        test_images = images[val_end:]
        
        # Copy to respective directories
        for split, img_list in [('train', train_images), 
                                 ('validation', val_images), 
                                 ('test', test_images)]:
            dest_dir = os.path.join(processed_dir, split, clean_name)
            for img in img_list:
                src = os.path.join(class_path, img)
                dst = os.path.join(dest_dir, img)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
            stats[split][clean_name] = len(img_list)
        
        print(f"  {clean_name}: train={len(train_images)}, val={len(val_images)}, test={len(test_images)}")
    
    return stats

# Create directories and split data
print("Creating split directories...")
create_split_directories(PROCESSED_DATA_DIR)

print("\nSplitting and copying data...")
split_stats = split_and_copy_data(
    RAW_DATA_DIR, 
    PROCESSED_DATA_DIR,
    train_ratio=CONFIG['train_split'],
    val_ratio=CONFIG['val_split']
)

print("\nData split complete!")

In [ ]:
# Verify split
print("\n" + "=" * 50)
print("SPLIT VERIFICATION")
print("=" * 50)

for split in ['train', 'validation', 'test']:
    split_path = os.path.join(PROCESSED_DATA_DIR, split)
    counts = count_images(split_path)
    total = sum(counts.values())
    print(f"\n{split.upper()}:")
    for class_name, count in counts.items():
        print(f"  {class_name}: {count}")
    print(f"  Total: {total}")

## 5. Data Augmentation Strategy

**Texture-Focused Augmentation for Coconut Mite Detection:**
- Heavy rotation (0-360°) - mite damage appears at any angle
- Zoom range - capture texture at different scales
- Brightness/contrast variation - lighting conditions vary
- Limited horizontal/vertical shift - keep fruit centered
- Color jitter - account for different coconut colors

In [ ]:
# Heavy augmentation for training - texture focused
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=360,  # Full rotation - mite patterns appear at any angle
    width_shift_range=0.15,  # Limited shift to keep fruit in frame
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=[0.8, 1.2],  # Zoom to capture texture at different scales
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='reflect',  # Reflect edges for natural look
    brightness_range=[0.7, 1.3],  # Account for lighting variation
    channel_shift_range=20.0,  # Color variation for different coconut types
)

# No augmentation for validation and test
val_test_datagen = ImageDataGenerator(rescale=1./255)

print("Data augmentation configured:")
print("  Training: Heavy augmentation (rotation, zoom, color, brightness)")
print("  Validation: Only rescaling")
print("  Test: Only rescaling")

In [ ]:
# Visualize augmentation
def show_augmented_images(generator, image_path, n_images=8):
    """Show augmented versions of a single image"""
    img = load_img(image_path, target_size=(CONFIG['img_size'], CONFIG['img_size']))
    x = img_to_array(img)
    x = x.reshape((1,) + x.shape)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    i = 0
    for batch in generator.flow(x, batch_size=1):
        axes[i].imshow(batch[0])
        axes[i].axis('off')
        axes[i].set_title(f'Augmented {i+1}')
        i += 1
        if i >= n_images:
            break
    
    plt.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

# Show augmentation for mite image
mite_sample = os.path.join(PROCESSED_DATA_DIR, 'train', 'mite', 
                           os.listdir(os.path.join(PROCESSED_DATA_DIR, 'train', 'mite'))[0])
fig = show_augmented_images(train_datagen, mite_sample)
plt.savefig(os.path.join(MODEL_DIR, 'augmentation_examples.png'), dpi=150)
plt.show()

## 6. Load Data Generators

In [ ]:
# Create data generators
train_generator = train_datagen.flow_from_directory(
    os.path.join(PROCESSED_DATA_DIR, 'train'),
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(PROCESSED_DATA_DIR, 'validation'),
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(PROCESSED_DATA_DIR, 'test'),
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=False
)

# Get class information
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

print(f"\nClasses: {class_names}")
print(f"Number of classes: {num_classes}")
print(f"Class indices: {train_generator.class_indices}")
print(f"\nTraining samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Test samples: {test_generator.samples}")

## 7. Class Weights

In [ ]:
# Calculate class weights for imbalanced dataset
train_labels = train_generator.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = dict(enumerate(class_weights_array))

print("Class Weights (to handle imbalance):")
print("=" * 40)
for idx, class_name in enumerate(class_names):
    count = np.sum(train_labels == idx)
    weight = class_weights[idx]
    print(f"  {class_name}: weight={weight:.4f} (count={count})")

print("\nHigher weight = less samples = model pays more attention")

## 8. Focal Loss

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    """
    Focal Loss for handling class imbalance
    - gamma: focusing parameter (higher = more focus on hard examples)
    - alpha: class weight balancing
    """
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        
        # Cross entropy
        cross_entropy = -y_true * tf.math.log(y_pred)
        
        # Focal weight
        weight = alpha * y_true * tf.pow(1 - y_pred, gamma)
        
        # Focal loss
        focal = weight * cross_entropy
        
        return tf.reduce_mean(tf.reduce_sum(focal, axis=-1))
    
    return focal_loss_fixed

print(f"Focal Loss configured with gamma={CONFIG['focal_gamma']}, alpha={CONFIG['focal_alpha']}")

## 9. Model Architecture - MobileNetV2 with Attention

Using MobileNetV2 which is excellent for:
- Texture pattern recognition
- Mobile deployment
- Efficient inference

Added attention mechanism to help model focus on damaged areas.

In [ ]:
def squeeze_excite_block(input_tensor, ratio=16):
    """Squeeze-and-Excitation block for channel attention"""
    filters = input_tensor.shape[-1]
    
    # Squeeze: Global Average Pooling
    se = layers.GlobalAveragePooling2D()(input_tensor)
    
    # Excitation: FC -> ReLU -> FC -> Sigmoid
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    
    # Reshape and multiply
    se = layers.Reshape((1, 1, filters))(se)
    
    return layers.Multiply()([input_tensor, se])


def create_model_with_attention(num_classes, l2_reg=0.01, dropout=0.4):
    """Create MobileNetV2-based model with attention mechanism"""
    
    # Base model (pre-trained on ImageNet)
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(CONFIG['img_size'], CONFIG['img_size'], 3)
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build model with attention
    inputs = keras.Input(shape=(CONFIG['img_size'], CONFIG['img_size'], 3))
    
    # Base model features
    x = base_model(inputs, training=False)
    
    # Add Squeeze-and-Excitation attention
    x = squeeze_excite_block(x, ratio=8)
    
    # Global pooling
    x = layers.GlobalAveragePooling2D()(x)
    
    # Dense layers with regularization
    x = layers.Dense(512, kernel_regularizer=l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout)(x)
    
    x = layers.Dense(256, kernel_regularizer=l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout)(x)
    
    # Output layer
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    
    return model, base_model

# Create model
model, base_model = create_model_with_attention(
    num_classes=num_classes,
    l2_reg=CONFIG['l2_reg'],
    dropout=CONFIG['dropout']
)

model.summary()

In [ ]:
# Compile model
model.compile(
    optimizer=Adam(learning_rate=CONFIG['learning_rate']),
    loss=focal_loss(gamma=CONFIG['focal_gamma'], alpha=CONFIG['focal_alpha']),
    metrics=['accuracy']
)

print("Model compiled with:")
print(f"  - Optimizer: Adam (lr={CONFIG['learning_rate']})")
print(f"  - Loss: Focal Loss (gamma={CONFIG['focal_gamma']}, alpha={CONFIG['focal_alpha']})")
print(f"  - Metrics: accuracy")

## 10. Callbacks

In [ ]:
# Callbacks for training
callbacks_list = [
    # Early stopping to prevent overfitting
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['patience'],
        min_delta=CONFIG['min_delta'],
        restore_best_weights=True,
        verbose=1
    ),
    
    # Model checkpoint - save best model
    callbacks.ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'best_model.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    
    # Reduce learning rate when stuck
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks configured:")
print(f"  - EarlyStopping: patience={CONFIG['patience']}")
print(f"  - ModelCheckpoint: saves best model")
print(f"  - ReduceLROnPlateau: reduces lr when stuck")

## 11. Phase 1: Train with Frozen Base

In [ ]:
print("="*60)
print("PHASE 1: Training with frozen base model")
print("="*60)
print(f"Training for up to {CONFIG['epochs']} epochs...")
print(f"Base model layers: FROZEN")
print()

history_phase1 = model.fit(
    train_generator,
    epochs=CONFIG['epochs'],
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=callbacks_list,
    verbose=1
)

print("\nPhase 1 training complete!")

## 12. Phase 2: Fine-tuning

In [ ]:
print("="*60)
print("PHASE 2: Fine-tuning with unfrozen base model")
print("="*60)

# Unfreeze top layers of base model
base_model.trainable = True

# Freeze early layers, unfreeze last 50 layers
for layer in base_model.layers[:-50]:
    layer.trainable = False

trainable_count = sum([1 for layer in base_model.layers if layer.trainable])
print(f"Unfrozen {trainable_count} layers in base model for fine-tuning")

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=CONFIG['learning_rate'] / 10),
    loss=focal_loss(gamma=CONFIG['focal_gamma'], alpha=CONFIG['focal_alpha']),
    metrics=['accuracy']
)

print(f"Learning rate reduced to {CONFIG['learning_rate'] / 10}")
print()

In [ ]:
# Continue training with fine-tuning
history_phase2 = model.fit(
    train_generator,
    epochs=40,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=callbacks_list,
    verbose=1
)

print("\nPhase 2 fine-tuning complete!")

## 13. Training History

In [ ]:
# Combine histories
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history.keys():
        combined[key] = h1.history[key] + h2.history[key]
    return combined

combined_history = combine_histories(history_phase1, history_phase2)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(combined_history['loss'], label='Training Loss', color='#2196F3', linewidth=2)
axes[0].plot(combined_history['val_loss'], label='Validation Loss', color='#F44336', linewidth=2)
axes[0].axvline(x=len(history_phase1.history['loss']), color='gray', linestyle='--', label='Fine-tuning Start')
axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(combined_history['accuracy'], label='Training Accuracy', color='#4CAF50', linewidth=2)
axes[1].plot(combined_history['val_accuracy'], label='Validation Accuracy', color='#FF9800', linewidth=2)
axes[1].axvline(x=len(history_phase1.history['accuracy']), color='gray', linestyle='--', label='Fine-tuning Start')
axes[1].set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_history.png'), dpi=150)
plt.show()

print(f"\nFinal Training Accuracy: {combined_history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {combined_history['val_accuracy'][-1]:.4f}")

## 14. Load Best Model

In [ ]:
# Load the best model
best_model_path = os.path.join(MODEL_DIR, 'best_model.keras')
custom_objects = {'focal_loss_fixed': focal_loss(CONFIG['focal_gamma'], CONFIG['focal_alpha'])}

best_model = keras.models.load_model(best_model_path, custom_objects=custom_objects)
print(f"Best model loaded from: {best_model_path}")

## 15. Test Set Evaluation

In [ ]:
print("="*60)
print("TEST SET EVALUATION (FINAL)")
print("="*60)

# Get predictions
test_generator.reset()
test_predictions = best_model.predict(test_generator, verbose=1)
test_pred_classes = np.argmax(test_predictions, axis=1)
test_true_classes = test_generator.classes

# Calculate metrics
test_accuracy = accuracy_score(test_true_classes, test_pred_classes)
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    test_true_classes, test_pred_classes, average='weighted'
)
test_macro_f1 = precision_recall_fscore_support(
    test_true_classes, test_pred_classes, average='macro'
)[2]

# ROC-AUC (for binary classification)
if num_classes == 2:
    test_roc_auc = roc_auc_score(test_true_classes, test_predictions[:, 1])
else:
    test_roc_auc = 'N/A (multi-class)'

print(f"\n{'Metric':<25} {'Value':<10}")
print("-" * 35)
print(f"{'Accuracy':<25} {test_accuracy:.4f}")
print(f"{'Precision (weighted)':<25} {test_precision:.4f}")
print(f"{'Recall (weighted)':<25} {test_recall:.4f}")
print(f"{'F1-Score (weighted)':<25} {test_f1:.4f}")
print(f"{'F1-Score (macro)':<25} {test_macro_f1:.4f}")
print(f"{'ROC-AUC':<25} {test_roc_auc if isinstance(test_roc_auc, str) else f'{test_roc_auc:.4f}'}")

In [ ]:
# Class-wise metrics
print("\n" + "="*60)
print("TEST SET - CLASS-WISE METRICS")
print("="*60)

test_report = classification_report(
    test_true_classes, 
    test_pred_classes, 
    target_names=class_names,
    digits=4,
    output_dict=True
)

print(classification_report(
    test_true_classes, 
    test_pred_classes, 
    target_names=class_names,
    digits=4
))

In [ ]:
# Class-wise metrics chart
fig, ax = plt.subplots(figsize=(10, 6))

metrics_df = pd.DataFrame({
    'Class': class_names,
    'Precision': [test_report[c]['precision'] for c in class_names],
    'Recall': [test_report[c]['recall'] for c in class_names],
    'F1-Score': [test_report[c]['f1-score'] for c in class_names]
})

x = np.arange(len(class_names))
width = 0.25

bars1 = ax.bar(x - width, metrics_df['Precision'], width, label='Precision', color='#2196F3', alpha=0.8)
bars2 = ax.bar(x, metrics_df['Recall'], width, label='Recall', color='#4CAF50', alpha=0.8)
bars3 = ax.bar(x + width, metrics_df['F1-Score'], width, label='F1-Score', color='#FF9800', alpha=0.8)

ax.set_ylabel('Score')
ax.set_title('Class-wise Metrics (Test Set)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([c.upper() for c in class_names])
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'class_metrics.png'), dpi=150)
plt.show()

## 16. Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_true_classes, test_pred_classes)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[c.upper() for c in class_names], 
            yticklabels=[c.upper() for c in class_names], ax=axes[0],
            annot_kws={'size': 16})
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# Normalized (percentages)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.1%', cmap='Greens', 
            xticklabels=[c.upper() for c in class_names], 
            yticklabels=[c.upper() for c in class_names], ax=axes[1],
            annot_kws={'size': 14})
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## 17. ROC Curve (Binary Classification)

In [ ]:
if num_classes == 2:
    # ROC Curve
    fpr, tpr, thresholds = roc_curve(test_true_classes, test_predictions[:, 1])
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color='#2196F3', lw=2, label=f'ROC Curve (AUC = {test_roc_auc:.4f})')
    ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Classifier')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curve - Mite Detection', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'roc_curve.png'), dpi=150)
    plt.show()
else:
    print("ROC curve skipped (multi-class classification)")

## 18. Sample Predictions

In [ ]:
# Show sample predictions
def show_predictions(generator, model, n_samples=8):
    """Show sample predictions with confidence"""
    generator.reset()
    batch_x, batch_y = next(generator)
    predictions = model.predict(batch_x, verbose=0)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for i in range(min(n_samples, len(batch_x))):
        ax = axes[i]
        ax.imshow(batch_x[i])
        
        true_class = class_names[np.argmax(batch_y[i])]
        pred_class = class_names[np.argmax(predictions[i])]
        confidence = np.max(predictions[i]) * 100
        
        color = 'green' if true_class == pred_class else 'red'
        ax.set_title(f'True: {true_class.upper()}\nPred: {pred_class.upper()} ({confidence:.1f}%)', 
                     color=color, fontsize=10, fontweight='bold')
        ax.axis('off')
    
    plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

fig = show_predictions(test_generator, best_model)
plt.savefig(os.path.join(MODEL_DIR, 'sample_predictions.png'), dpi=150)
plt.show()

## 19. Save Model Info

In [ ]:
# Save model info
model_info = {
    'version': 'v12',
    'model_name': 'coconut_mite_v12_fruit_focused',
    'architecture': 'MobileNetV2 + Squeeze-Excite Attention',
    'input_shape': [CONFIG['img_size'], CONFIG['img_size'], 3],
    'num_classes': num_classes,
    'class_names': class_names,
    'class_indices': train_generator.class_indices,
    'training_config': CONFIG,
    'focus': 'Coconut fruit surface patterns - mite damage detection',
    'metrics': {
        'test_accuracy': float(test_accuracy),
        'test_precision': float(test_precision),
        'test_recall': float(test_recall),
        'test_f1_score': float(test_f1),
        'test_macro_f1': float(test_macro_f1),
        'test_roc_auc': float(test_roc_auc) if isinstance(test_roc_auc, float) else test_roc_auc,
        'class_wise': {
            c: {
                'precision': float(test_report[c]['precision']),
                'recall': float(test_report[c]['recall']),
                'f1-score': float(test_report[c]['f1-score']),
                'support': int(test_report[c]['support'])
            } for c in class_names
        }
    },
    'dataset': {
        'source': 'mite - new',
        'train_samples': train_generator.samples,
        'validation_samples': val_generator.samples,
        'test_samples': test_generator.samples,
    },
    'trained_at': datetime.now().isoformat(),
    'notes': 'Binary classification model focused on coconut fruit surface patterns. Uses MobileNetV2 with attention mechanism for texture recognition.'
}

# Save to JSON
info_path = os.path.join(MODEL_DIR, 'model_info.json')
with open(info_path, 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"Model info saved to: {info_path}")

In [ ]:
# List all saved files
print("\n" + "="*60)
print("SAVED FILES")
print("="*60)

for f in os.listdir(MODEL_DIR):
    filepath = os.path.join(MODEL_DIR, f)
    size = os.path.getsize(filepath) / (1024 * 1024)  # MB
    print(f"  {f}: {size:.2f} MB")

## 20. Final Summary

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE - COCONUT MITE DETECTION v12")
print("="*60)

print(f"\nModel: Coconut Mite Detection v12 (Fruit Surface Focused)")
print(f"Architecture: MobileNetV2 + Squeeze-Excite Attention")
print(f"Classes: {[c.upper() for c in class_names]}")

print(f"\n{'TEST SET RESULTS':^40}")
print("-"*40)
print(f"{'Accuracy:':<20} {test_accuracy*100:.2f}%")
print(f"{'Precision:':<20} {test_precision*100:.2f}%")
print(f"{'Recall:':<20} {test_recall*100:.2f}%")
print(f"{'F1-Score:':<20} {test_f1*100:.2f}%")
print(f"{'Macro F1:':<20} {test_macro_f1*100:.2f}%")
if isinstance(test_roc_auc, float):
    print(f"{'ROC-AUC:':<20} {test_roc_auc*100:.2f}%")

print(f"\n{'CLASS-WISE F1-SCORES':^40}")
print("-"*40)
for c in class_names:
    print(f"{c.upper():<20} {test_report[c]['f1-score']*100:.2f}%")

print(f"\nModel saved to: {MODEL_DIR}")
print("="*60)

## 21. Compare with v10 (if exists)

In [ ]:
# Try to load v10 model info for comparison
v10_info_path = os.path.join(BASE_DIR, 'models', 'coconut_mite_v10', 'model_info.json')

if os.path.exists(v10_info_path):
    with open(v10_info_path, 'r') as f:
        v10_info = json.load(f)
    
    print("\n" + "="*60)
    print("COMPARISON: v12 vs v10")
    print("="*60)
    
    print(f"\n{'Metric':<25} {'v10':<15} {'v12':<15} {'Diff':<10}")
    print("-" * 65)
    
    v10_acc = v10_info['metrics']['test_accuracy']
    v10_f1 = v10_info['metrics']['test_f1_score']
    
    print(f"{'Accuracy':<25} {v10_acc*100:.2f}%{'':<7} {test_accuracy*100:.2f}%{'':<7} {(test_accuracy-v10_acc)*100:+.2f}%")
    print(f"{'F1-Score':<25} {v10_f1*100:.2f}%{'':<7} {test_f1*100:.2f}%{'':<7} {(test_f1-v10_f1)*100:+.2f}%")
    
    print("\nv12 Improvements:")
    print("  - Binary classification (mite vs healthy) - no not_coconut class")
    print("  - New high-quality dataset with clear fruit patterns")
    print("  - MobileNetV2 + Attention for texture recognition")
    print("  - Heavy augmentation for fruit surface patterns")
else:
    print("\nv10 model info not found for comparison")